In [5]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA GeForce RTX 4070 Ti


In [2]:
cd ice_sleep_detpj/

/home/jovyan/ice_sleep_detpj


In [48]:
ls

 CITATION.cff        benchmarks.py         ice_sleep_detpj/   train.py
 CONTRIBUTING.md     classify/             models/            tutorial.ipynb
 LICENSE             copy_txt_to_yolo.py   pyproject.toml     utils/
 README.md           data/                 requirements.txt   val.py
 README.zh-CN.md     detect.py             runs/              yolov5s.pt
'Untitled Folder'/   export.py             segment/
 __pycache__/        hubconf.py            train.ipynb


In [8]:
!find . -name "*ice_driver.yaml"

./data/ice_driver.yaml


In [3]:
!find . -name "*train.py"

./ice_sleep_detpj/utils/train.py
./ice_sleep_detpj/segment/train.py
./ice_sleep_detpj/classify/train.py
./ice_sleep_detpj/train.py


In [57]:
!unzip -q /home/jovyan/ice_sleep_detpj/data/YOLO_dataset_NOHEAD.zip -d /home/jovyan/ice_sleep_detpj/data/ice_th


In [32]:
!cat /home/jovyan/ice_sleep_detpj/data/ice_driver.yaml


train: /home/jovyan/ice_sleep_detpj/data/YOLO_dataset/images/train
val: /home/jovyan/ice_sleep_detpj/data/YOLO_dataset/images/val # 따로 없으면 train 재사용


nc: 6
names: [
  'Leye_Open',
  'Leye_Closed',
  'Reye_Open',
  'Reye_Closed',
  'Mouth_Open',
  'Mouth_Closed'
]



In [1]:
!pip install opencv-python

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 112.3 MB/s eta 0:00:0000:0100:01


In [59]:
rm -rf /home/jovyan/ice_sleep_detpj/data/ice_th/YOLO_dataset/labels/val_txt


In [30]:
rm data/ice_texi_train/labels/train.cache.npy

In [ ]:
import json
import os

def convert_bbox(bbox, img_width, img_height):
    x_min, y_min, x_max, y_max = map(float, bbox)
    x_center = (x_min + x_max) / 2 / img_width
    y_center = (y_min + y_max) / 2 / img_height
    w = (x_max - x_min) / img_width
    h = (y_max - y_min) / img_height
    return x_center, y_center, w, h

def merge_bboxes(b1, b2):
    x_min = min(float(b1[0]), float(b2[0]))
    y_min = min(float(b1[1]), float(b2[1]))
    x_max = max(float(b1[2]), float(b2[2]))
    y_max = max(float(b1[3]), float(b2[3]))
    return [x_min, y_min, x_max, y_max]

def json_to_yolo(json_path, output_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    width = int(data['FileInfo']['Width'])
    height = int(data['FileInfo']['Height'])
    bboxes = data['ObjectInfo']['BoundingBox']

    result_lines = []

    # 👁️ 눈 (양쪽 눈 기준 합쳐서)
    leye = bboxes.get('Leye')
    reye = bboxes.get('Reye')
    if leye['isVisible'] and reye['isVisible']:
        is_open = leye['Opened'] or reye['Opened']
        class_id = 0 if is_open else 1  # 0: eye_open, 1: eye_closed
        merged_box = merge_bboxes(leye['Position'], reye['Position'])
        coords = convert_bbox(merged_box, width, height)
        result_lines.append(f"{class_id} {' '.join(map(str, coords))}")

    # 👄 입
    mouth = bboxes.get('Mouth')
    if mouth['isVisible']:
        class_id = 2 if mouth['Opened'] else 3  # 2: mouth_open, 3: mouth_closed
        coords = convert_bbox(mouth['Position'], width, height)
        result_lines.append(f"{class_id} {' '.join(map(str, coords))}")

    # 저장
    base_name = os.path.splitext(os.path.basename(json_path))[0]
    output_file = os.path.join(output_path, base_name + '.txt')
    with open(output_file, 'w') as f:
        f.write('\n'.join(result_lines))

# 📁 Jupyter 환경 기준 경로 (수정됨)
input_dir = 'data/ice_th/YOLO_dataset/labels/train'        # JSON 파일들
output_dir = 'data/ice_th/YOLO_dataset/labels/train_txt'   # YOLO txt 저장 경로

os.makedirs(output_dir, exist_ok=True)

# 변환 실행
for filename in os.listdir(input_dir):
    if filename.endswith('.json'):
        json_path = os.path.join(input_dir, filename)
        json_to_yolo(json_path, output_dir)

print("✅ 변환 완료!")


In [31]:
!python train.py --img 640 --batch 16 --epochs 30 --data data/ice_texi.yaml --weights yolov5s.pt --name ice_texi_mouth --workers 8


2025-05-07 17:53:49.597967: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-07 17:53:49.597995: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-07 17:53:49.598524: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
train: weights=yolov5s.pt, cfg=, data=data/ice_texi.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=30, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=, multi_scale=False, single_cls=Fal

In [64]:
!python detect.py \
  --weights runs/train/ice_eye_mouth20/weights/best.pt \
  --source data/ice_testdata
  --img 640 \
  --conf-thres 0.65 \
  --save-txt \
  --save-conf \
  --project runs/detect \
  --name ice_yolo_test20_3 \
  --exist-ok



detect: weights=['runs/train/ice_eye_mouth20/weights/best.pt'], source=data/ice_testdata, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.65, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=True, save_format=0, save_csv=False, save_conf=True, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=ice_yolo_test20_3, exist_ok=True, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-417-g9e6d7c1b Python-3.11.9 torch-2.2.2+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 12010MiB)

Fusing layers... 
Model summary: 157 layers, 7026307 parameters, 0 gradients, 15.8 GFLOPs
image 1/3200 /home/jovyan/ice_sleep_detpj/data/ice_testdata/R_332_50_M_01_M0_G1_C0_01.jpg: 640x384 1 Leye_Open, 1 Reye_Open, 1 Mouth_Closed, 36.7ms
image 2/3200 /home/jovyan/ice_sleep_detpj/data/ice_testdata/R_332_50_M_01_M0_G1_C0_02.jpg: 640x384 1 Leye_Open, 1 Reye_Open, 1 M